# IMERG half-hourly point time series with Dask

Each half-hourly time step is one on-disk chunk covering the full 3600 x 1800 grid, so extracting one
point still means reading and decompressing every time step's whole field (~444,000 of them for 2000-2025).
This notebook spreads those reads across Dask workers on Casper, then plots the result.

In [ ]:
import time
import os
import xarray as xr
import matplotlib.pyplot as plt
from dask.distributed import Client, progress
from dask_jobqueue import PBSCluster

## Settings

In [ ]:
# Point this at the combined 2000-2025 half-hourly reference file
#data_file = '/glade/u/home/bonnland/GitRepos/gdex-arco-kerchunk/examples/convert_d731000/precipitation_GPM_3IMERGHH_v07.parq'
data_file = '/glade/derecho/scratch/bonnland/IMERG_d731000_NEW/precipitation_GPM_3IMERGHH_v07.parq'

lat, lon = 39.7, -104.9   # e.g. Denver
variable = "precipitation"
n_workers = 32            # ~444k steps x ~0.13 s / 64 workers = ~15 min of reading

## Start a Dask cluster on Casper

One single-core PBS job per worker; workers use the same Python as this kernel.
If workers start but never connect, try `interface="ib0"`. If many jobs sit in the queue, lower `n_workers`.

In [ ]:
user = os.environ["USER"]
log_dir = f"/glade/derecho/scratch/{user}/dask/logs"
os.makedirs(log_dir, exist_ok=True)

cluster = PBSCluster(
    account="STDD0003",
    queue="casper",
    cores=1,
    processes=1,
    memory="4GB",
    resource_spec="select=1:ncpus=1:mem=4GB",
    walltime="02:00:00",
    interface="ext",
    local_directory="$TMPDIR",
    log_directory=log_dir,
)
print(cluster.job_script())

In [ ]:
cluster.scale(jobs=n_workers)
client = Client(cluster)
client

In [ ]:
client.wait_for_workers(n_workers // 2, timeout=900)
print(f"{len(client.scheduler_info()['workers'])} workers connected")

## Read and plot

In [ ]:
def plot_timeseries(ref_path, lat, lon, variable="precipitation",
                    remote_protocol="file", time_block=100, ax=None):
    """Read the time series nearest lat/lon in parallel with Dask, then plot it.

    Returns (ax, series), where series is the loaded 1-D DataArray.
    """
    ds = xr.open_dataset(ref_path, engine="kerchunk",
                         storage_options={"remote_protocol": remote_protocol})
    if variable not in ds.data_vars:
        raise KeyError(f"'{variable}' not found. Available: {list(ds.data_vars)}")

    # Some combined references store lat/lon as (time, lat) / (time, lon).
    # Restore 1-D index coordinates so .sel(method="nearest") works.
    for c in ("lat", "lon"):
        if ds[c].ndim > 1:
            ds = ds.assign_coords({c: (c, ds[c].isel(time=0).values, ds[c].attrs)})

    # Dask chunks: on-disk shape in lat/lon, `time_block` steps per task
    da = ds[variable]
    disk = da.encoding.get("chunks")
    chunks = dict(zip(da.dims, disk)) if disk else {d: -1 for d in da.dims}
    t_disk = chunks.get("time", 1)
    chunks["time"] = max(t_disk, (time_block // t_disk) * t_disk)
    point = da.chunk(chunks).sel(lat=lat, lon=lon, method="nearest")
    print(f"{point.sizes['time']:,} time steps in {point.data.npartitions:,} tasks")

    # Read in parallel on the cluster, with a progress bar
    t0 = time.perf_counter()
    point = point.persist()
    progress(point)
    series = point.compute()
    print(f"read in {time.perf_counter() - t0:.1f} s")

    # Plot (no markers, so 400k+ points draw quickly)
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 4))
    series.plot.line(ax=ax, x="time", linewidth=0.5)
    ax.set_title(f"{variable} at ({float(series.lat):.2f}\u00b0, {float(series.lon):.2f}\u00b0)\n"
                 f"(nearest to requested {lat}\u00b0, {lon}\u00b0)")
    units = series.attrs.get("units") or series.attrs.get("Units", "")
    ax.set_ylabel(variable + (f" ({units})" if units else ""))
    ax.set_xlabel("Time")
    ax.grid(True, alpha=0.3)
    return ax, series

In [ ]:
ax, series = plot_timeseries(data_file, lat, lon, variable)
plt.show()

In [ ]:
# Optional: daily totals are easier to read at this length (half-hourly mm/hr -> mm/day)
# (series * 0.5).resample(time="1D").sum().plot.line(x="time", figsize=(12, 4))

# Optional: save the series so it never has to be read again
# series.to_netcdf("precip_point_halfhourly.nc")

## Shut down

In [ ]:
client.close()
cluster.close()